In [27]:
import pandas as pd

# Load CSVs 
orders = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
geolocation = pd.read_csv("olist_geolocation_dataset.csv")
translation = pd.read_csv("product_category_name_translation.csv")

# --- Merge step 1: Customers + Orders ---
cust_orders = pd.merge(orders, customers, on="customer_id", how="left")

# --- Merge step 2: Add Order Items ---
cust_orders_items = pd.merge(cust_orders, order_items, on="order_id", how="left")

# --- Merge step 3: Add Product Details ---
cust_orders_items_products = pd.merge(
    cust_orders_items, products[[
        "product_id", "product_category_name"]],
    on="product_id", how="left"
)
    
cust_orders_items_products_translated = pd.merge(
   cust_orders_items_products, translation, 
    how="left", 
    on="product_category_name"
)
# --- Merge step 4: Add Review Scores ---
cust_orders_items_products_reviews = pd.merge(
    cust_orders_items_products_translated,
    reviews[["order_id", "review_score"]],
    on="order_id", how="left"
)
payments_agg = (
    payments.groupby("order_id").agg({
        "payment_type": "first",          # take the first payment type per order
        "payment_installments": "first",  # take the first installment value per order
        "payment_value": "sum"            # optional: sum the total amount paid
    }).reset_index()
)
# --- Merge step 5: Add Payment Details ---
cust_orders_items_products_reviews_pay = pd.merge(
    cust_orders_items_products_reviews,
    payments_agg[["order_id", "payment_type", "payment_installments"]],
    on="order_id", how="left"
)

# --- Merge step 6: Add Geolocation (by ZIP prefix) ---
# Average lat/lng per zipcode prefix to reduce duplicates
geo_avg = geolocation.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].mean().reset_index()

full_data = pd.merge(
    cust_orders_items_products_reviews_pay,
    geo_avg,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)
# --- Select columns ---
full_data = full_data[[
    "order_id", "customer_id", "customer_unique_id",
    "customer_city", "customer_state",
    "geolocation_lat", "geolocation_lng",
    "order_status", "order_delivered_customer_date",
    "product_id", "product_category_name_english",
    "price",
    "review_score",
    "payment_type", "payment_installments"
]]



print(full_data.head())
print("Final shape:", full_data.shape)


                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

                 customer_unique_id            customer_city customer_state  \
0  7c396fd4830fd04220f754e42b4e5bff                sao paulo             SP   
1  af07308b275d755c9edb36a90c618231                barreiras             BA   
2  3a653a41f6f9fc3d2a113cf8398680e8               vianopolis             GO   
3  7c142cf63193a1473d2e66489a9ae977  sao goncalo do amarante             RN   
4  72632f0f9dd73dfee390c9b22eb56dd6              santo andre             SP   

   geolocation_lat  geolocation_lng order_status  \
0       -23.576983       -46.58716

In [28]:
full_data=full_data.drop_duplicates()

In [29]:
#  keep only orders that were delivered
full_data= full_data[full_data['order_status']=='delivered']

In [32]:
print("Final shape:", full_data.shape)

Final shape: (100410, 15)


In [33]:
full_data["order_delivered_customer_date"] = pd.to_datetime(full_data["order_delivered_customer_date"], errors="coerce")
order_level = (
    full_data.groupby("order_id").agg({
        "customer_unique_id": "first",
        "price": ["sum", "mean"],                 # total order value + avg item price
        "payment_installments": "max",
        "review_score": "mean",
        "order_delivered_customer_date": "max",
        "customer_city": "first",
        "customer_state": "first",
        "geolocation_lat": "mean",
        "geolocation_lng": "mean",
        "product_id": "count"                     # number of items in order
    }).reset_index()
)
# Flatten multi-level column names
order_level.columns = [
    "order_id", "customer_unique_id",
    "order_price_total", "order_avg_item_price", "payment_installments",
    "review_score", "order_delivered_customer_date",
    "customer_city", "customer_state",
    "geolocation_lat", "geolocation_lng",
    "num_items_in_order"
]
customer_behavioral = (
    order_level.groupby("customer_unique_id").agg({
        "order_id": "nunique",
        "order_price_total": ["sum", "mean"],       # total & avg order value
        "order_avg_item_price": "mean",             # avg item price per order
        "num_items_in_order": "mean",               # avg items per order
        "review_score": "mean",
        "payment_installments": "mean",
        "order_delivered_customer_date": "max",
        "customer_city": "first",
        "customer_state": "first",
        "geolocation_lat": "mean",
        "geolocation_lng": "mean"
    }).reset_index()
)
# Flatten multi-level columns
customer_behavioral.columns = [
    "customer_unique_id", "num_orders",
    "total_spent", "avg_order_value",
    "avg_item_price_per_order", "avg_items_per_order",
    "avg_review", "avg_installments",
    "last_purchase_date", "city", "state",
    "latitude", "longitude"
]

customer_behavioral["recency_days"] = (
    customer_behavioral["last_purchase_date"].max() - customer_behavioral["last_purchase_date"]
).dt.days

num_categories = (
    full_data.groupby("customer_unique_id")["product_category_name_english"]
    .nunique()
    .reset_index(name="num_product_categories")
)
category_counts = (
    full_data.groupby(["customer_unique_id", "product_category_name_english"])
    .size()
    .reset_index(name="purchase_count")
)

most_bought = (
    category_counts.loc[
        category_counts.groupby("customer_unique_id")["purchase_count"].idxmax(),
        ["customer_unique_id", "product_category_name_english"]
    ].rename(columns={"product_category_name_english": "most_bought_category"})
)

least_bought = (
    category_counts.loc[
        category_counts.groupby("customer_unique_id")["purchase_count"].idxmin(),
        ["customer_unique_id", "product_category_name_english"]
    ].rename(columns={"product_category_name_english": "least_bought_category"})
)

customer_pref = most_bought.merge(least_bought, on="customer_unique_id", how="left")
payment_counts = (
    full_data.groupby(["customer_unique_id", "payment_type"])
    .size()
    .reset_index(name="payment_count")
)

most_payment = (
    payment_counts.loc[
        payment_counts.groupby("customer_unique_id")["payment_count"].idxmax(),
        ["customer_unique_id", "payment_type"]
    ].rename(columns={"payment_type": "most_used_payment"})
)

least_payment = (
    payment_counts.loc[
        payment_counts.groupby("customer_unique_id")["payment_count"].idxmin(),
        ["customer_unique_id", "payment_type"]
    ].rename(columns={"payment_type": "least_used_payment"})
)

payment_pref = most_payment.merge(least_payment, on="customer_unique_id", how="left")

customer_data = (
    customer_behavioral
    .merge(num_categories, on="customer_unique_id", how="left")
    .merge(customer_pref, on="customer_unique_id", how="left")
    .merge(payment_pref, on="customer_unique_id", how="left")
)

In [34]:
print(customer_data.head())
print("Final shape:", customer_data.shape)

                 customer_unique_id  num_orders  total_spent  avg_order_value  \
0  0000366f3b9a7992bf8c76cfdf3221e2           1       129.90           129.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f           1        18.90            18.90   
2  0000f46a3911fa3c0805444483337064           1        69.00            69.00   
3  0000f6ccb0745a6a4b88665a16c9f078           1        25.99            25.99   
4  0004aac84e0df4da2b147fca70cf8255           1       180.00           180.00   

   avg_item_price_per_order  avg_items_per_order  avg_review  \
0                    129.90                  1.0         5.0   
1                     18.90                  1.0         4.0   
2                     69.00                  1.0         3.0   
3                     25.99                  1.0         4.0   
4                    180.00                  1.0         5.0   

   avg_installments  last_purchase_date      city state   latitude  longitude  \
0               8.0 2018-05-16 20:48:37   cajam

In [37]:
# Save combined dataset 
customer_data.to_csv("olist_customer_base_dataset.csv", index=False)